<a href="https://colab.research.google.com/github/IvanMorsin/Forecasting-electrical-power-in-multi-storey-residential-buildings/blob/main/notebook_7_classical_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kaleido==0.2.1 workalendar -q

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from workalendar.europe import Russia
from tqdm import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

In [3]:
def save_predictions(ts, house_ids, y_true, y_pred, horizon_name, model_name, filename):
    df_pred = pd.DataFrame({
        'timestamp': ts,
        'house_id': house_ids,
        'y_true': y_true,
        'y_pred': y_pred,
        'horizon': horizon_name,
        'model': model_name,
    })
    if os.path.exists(filename):
        df_existing = pd.read_csv(filename)
        df_pred = pd.concat([df_existing, df_pred], ignore_index=True)
    df_pred.to_csv(filename, index=False)

In [4]:
df = pd.read_csv('df_final+whether.csv', parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

HOUSE_META = {
    'house_1': {'n_flats': 383, 'n_floors': 12},
    'house_2': {'n_flats': 191, 'n_floors': 12},
    'house_3': {'n_flats': 124, 'n_floors': 12},
    'house_4': {'n_flats': 263, 'n_floors': 12},
    'house_5': {'n_flats': 127, 'n_floors':  7},
    'house_6': {'n_flats': 497, 'n_floors': 25},
    'house_7': {'n_flats': 471, 'n_floors': 17},
    'house_8': {'n_flats': 171, 'n_floors': 23},
}
HOUSES = list(HOUSE_META.keys())

HORIZONS = {
    '4h':  8,
    '8h':  16,
    '24h': 48,
    '7d':  336,
    '14d': 672,
    '1m':  1488,
}

cal = Russia()
def is_holiday(dt):
    if dt.weekday() >= 5:
        return 0
    return int(not cal.is_working_day(dt.date()))

df['is_holiday'] = df['timestamp'].apply(is_holiday)

In [5]:
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    return {"MAE": round(mae, 3), "RMSE": round(rmse, 3), "MAPE": round(mape, 3)}

# Количество точек в горизонте прогнозов
horizons = {
    "4h":  8,
    "8h":  16,
    "24h": 48,
    "7d":  336,
    "14d": 672,
    "1m":  1488,
}

In [6]:
# длинный формат - все дома
LAG_SEASON = 336

def make_features_all(df):
    frames = []
    for house, meta in HOUSE_META.items():
        data = df[['timestamp', house]].copy()
        data = data.rename(columns={house: 'power'})
        data['hour'] = data['timestamp'].dt.hour
        data['minute'] = data['timestamp'].dt.minute
        data['weekday'] = data['timestamp'].dt.weekday
        data['month'] = data['timestamp'].dt.month
        data['day_of_year'] = data['timestamp'].dt.dayofyear
        data['is_weekend'] = (data['weekday'] >= 5).astype(int)
        data['is_holiday'] = df['is_holiday'].values

        for lag in [1, 2, 48, 96, 336, 672, 1488]:
            data[f'lag_{lag}'] = data['power'].shift(lag)

        data['rolling_mean_48'] = data['power'].shift(1).rolling(48).mean()
        data['rolling_mean_336'] = data['power'].shift(1).rolling(336).mean()
        data['rolling_mean_672'] = data['power'].shift(1).rolling(672).mean()
        data['rolling_mean_1488'] = data['power'].shift(1).rolling(1488).mean()

        data['temp_c'] = df['temp_c'].values
        data['humidity'] = df['humidity'].values
        data['cloudiness'] = df['cloudiness'].values

        data['n_flats'] = meta['n_flats']
        data['n_floors'] = meta['n_floors']

        data['month_sin'] = np.sin(2 * np.pi * data['month'] / 12)
        data['month_cos'] = np.cos(2 * np.pi * data['month'] / 12)

        data['week_of_year'] = data['timestamp'].dt.isocalendar().week
        max_week = 52.0
        data['week_norm'] = (data['week_of_year'] - 1) / max_week  # от 0 до 1
        data['week_sin'] = np.sin(2 * np.pi * data['week_norm'])
        data['week_cos'] = np.cos(2 * np.pi * data['week_norm'])

        data['house_id'] = house
        frames.append(data)

    df_all = pd.concat(frames, ignore_index=True)
    df_all = df_all.sort_values(['timestamp', 'house_id']).reset_index(drop=True)
    df_all = df_all.dropna().reset_index(drop=True)
    return df_all

df_long = make_features_all(df)

feature_cols_num = [c for c in df_long.columns
                    if c not in ['timestamp', 'power', 'house_id']]
feature_cols_all = feature_cols_num + ['house_id']
cat_idx = len(feature_cols_num)

# split по времени
split_train = df['timestamp'].quantile(0.70)
split_val   = df['timestamp'].quantile(0.85)

# фолды по уникальным timestamps
unique_ts = df['timestamp'].values
tscv = TimeSeriesSplit(n_splits=5)
folds = []
for train_idx, val_idx in tscv.split(unique_ts):
    folds.append((unique_ts[train_idx[-1]], unique_ts[val_idx[-1]]))
print(f'Признаки: {feature_cols_num}')

Признаки: ['hour', 'minute', 'weekday', 'month', 'day_of_year', 'is_weekend', 'is_holiday', 'lag_1', 'lag_2', 'lag_48', 'lag_96', 'lag_336', 'lag_672', 'lag_1488', 'rolling_mean_48', 'rolling_mean_336', 'rolling_mean_672', 'rolling_mean_1488', 'temp_c', 'humidity', 'cloudiness', 'n_flats', 'n_floors', 'month_sin', 'month_cos', 'week_of_year', 'week_norm', 'week_sin', 'week_cos']


In [7]:
# Naive модель: прогноз = последнее известное значение
# для горизонта из N точек просто повторяем последнее значение N раз

naive_rows = []

for house in HOUSES:
    for horizon_name, horizon_points in horizons.items():
        fold_metrics = []
        df_house = df_long[df_long['house_id'] == house].sort_values(
            'timestamp').reset_index(drop=True)

        for t_train_end, t_val_end in folds:
            last_val = df_house.loc[
                df_house['timestamp'] <= t_train_end, 'power'].iloc[-1]
            y_true = df_house.loc[
                df_house['timestamp'] > t_train_end, 'power'
            ].values[:horizon_points]

            if len(y_true) < horizon_points:
                continue

            y_pred = np.full(horizon_points, last_val)
            fold_metrics.append(evaluate(y_true, y_pred))

        if fold_metrics:
            naive_rows.append({
                'Модель': 'Naive', 'Дом': house, 'Горизонт': horizon_name,
                'MAE':  round(np.mean([m['MAE']  for m in fold_metrics]), 3),
                'RMSE': round(np.mean([m['RMSE'] for m in fold_metrics]), 3),
                'MAPE': round(np.mean([m['MAPE'] for m in fold_metrics]), 3),
            })

df_naive = pd.DataFrame(naive_rows)
df_naive.to_csv('results_naive.csv', index=False)

pivot = df_naive.pivot(index='Горизонт', columns='Дом', values='MAPE')
pivot = pivot.reindex(list(horizons.keys()))
pivot['Среднее'] = pivot.mean(axis=1).round(2)
print('Naive: MAPE:')
print(pivot.round(2).to_string())

Naive: MAPE:
Дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
Горизонт                                                                                 
4h          20.40    19.28    20.25    17.40    14.03    15.13    18.70    14.25    17.43
8h          36.19    37.74    37.28    32.32    27.06    30.34    38.02    30.51    33.68
24h         35.19    38.72    38.10    32.11    29.62    34.03    36.65    35.05    34.93
7d          38.36    41.19    38.99    33.51    32.12    37.40    38.94    36.92    37.18
14d         38.69    40.98    38.53    33.10    32.20    37.09    38.67    36.82    37.01
1m          39.23    42.54    38.22    33.79    33.13    37.27    39.16    38.44    37.72


In [8]:
train_idx, val_idx = list(tscv.split(df))[-1]
house = "house_1"

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f"Горизонт {h}" for h in horizons.keys()],
    vertical_spacing=0.1,
    horizontal_spacing=0.08
)

for i, (horizon_name, horizon_points) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    last_value = df[house].iloc[train_idx[-1]]
    y_pred = np.full(horizon_points, last_value)
    y_true = df[house].iloc[val_idx[:horizon_points]].values
    timestamps = df["timestamp"].iloc[val_idx[:horizon_points]]

    fig.add_trace(go.Scatter(
        x=timestamps, y=y_true,
        mode="lines", name="Фактические значения",
        line=dict(color="#1f77b4", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=timestamps, y=y_pred,
        mode="lines", name="прогноз Naive",
        line=dict(color="#d62728", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

fig.update_layout(
    title="Naive: Прогнозные и фактические значения по всем горизонтам прогнозирования (дом 1, fold 5)",
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9))
)

fig.show()
fig.write_image("27b_naive_forecast_all_horizons.png", scale=2)

Наглядно видно слабость Naive — прямая горизонтальная линия совершенно не улавливает никакой график

In [9]:
# Seasonal Naive: прогноз значения электрической нагрузки того же времени неделю назад
# лаг 336 = 7 дней * 48 интервалов

seasonal_rows = []

for house in HOUSES:
    df_house = df_long[df_long['house_id'] == house].sort_values(
        'timestamp').reset_index(drop=True)

    for horizon_name, horizon_points in horizons.items():
        fold_metrics = []

        for t_train_end, t_val_end in folds:
            train_vals = df_house.loc[
                df_house['timestamp'] <= t_train_end, 'power'].values
            val_vals = df_house.loc[
                df_house['timestamp'] > t_train_end, 'power'].values

            if len(train_vals) < LAG_SEASON + horizon_points:
                continue
            if len(val_vals) < horizon_points:
                continue

            y_season = train_vals[-LAG_SEASON:]
            repeats  = (horizon_points // LAG_SEASON) + 1
            y_pred   = np.tile(y_season, repeats)[:horizon_points]
            y_true = val_vals[:horizon_points]

            if len(y_pred) != horizon_points or len(y_true) != horizon_points:
                continue

            fold_metrics.append(evaluate(y_true, y_pred))

        if fold_metrics:
            seasonal_rows.append({
                'Модель': 'Seasonal Naive', 'Дом': house, 'Горизонт': horizon_name,
                'MAE': round(np.mean([m['MAE']  for m in fold_metrics]), 3),
                'RMSE': round(np.mean([m['RMSE'] for m in fold_metrics]), 3),
                'MAPE': round(np.mean([m['MAPE'] for m in fold_metrics]), 3),
            })

df_seasonal_naive = pd.DataFrame(seasonal_rows)
df_seasonal_naive.to_csv('results_seasonal_naive.csv', index=False)

pivot = df_seasonal_naive.pivot(index='Горизонт', columns='Дом', values='MAPE')
pivot = pivot.reindex(list(horizons.keys()))
pivot['Среднее'] = pivot.mean(axis=1).round(2)
print('Seasonal Naive: MAPE:')
print(pivot.round(2).to_string())

Seasonal Naive: MAPE:
Дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
Горизонт                                                                                 
4h           5.82    10.79     8.41     6.11    10.41     6.04     4.46     7.43     7.43
8h           6.15     9.86     8.10     6.68     9.05     7.20     4.76     5.95     7.22
24h          5.56     9.22     9.18     6.68     8.38     6.73     6.38     6.24     7.30
7d           6.64     8.64    10.60     6.58     8.51     6.70     6.43     7.06     7.65
14d          6.61     8.68    10.42     6.86     8.90     7.27     6.29     6.95     7.75
1m           7.01     9.01    10.67     8.10     9.76     8.10     6.52     8.04     8.40


In [10]:
train_idx, val_idx = list(tscv.split(df))[-1]
house = "house_1"
lag = 336

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f"Горизонт {h}" for h in horizons.keys()],
    vertical_spacing=0.1,
    horizontal_spacing=0.08
)

for i, (horizon_name, horizon_points) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    start_idx = val_idx[0]
    y_pred = df[house].iloc[start_idx - lag: start_idx - lag + horizon_points].values
    y_true = df[house].iloc[val_idx[:horizon_points]].values
    timestamps = df["timestamp"].iloc[val_idx[:horizon_points]]

    fig.add_trace(go.Scatter(
        x=timestamps, y=y_true,
        mode="lines", name="факт",
        line=dict(color="#1f77b4", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=timestamps, y=y_pred,
        mode="lines", name="прогноз Seasonal Naive",
        line=dict(color="#d62728", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

fig.update_layout(
    title="Seasonal Naive: Прогнозные и фактические значения по всем горизонтам прогнозирования (дом 1, fold 5)",
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9))
)

fig.show()
fig.write_image("28b_seasonal_naive_forecast_all_horizons.png", scale=2)

In [11]:
lr_rows = []

for horizon_name, horizon_points in tqdm(HORIZONS.items(), desc='LR'):

    frames_shifted = []
    for house in HOUSES:
        df_h = df_long[df_long['house_id'] == house].copy()
        df_h['power_target'] = df_h['power'].shift(-horizon_points)
        frames_shifted.append(df_h)

    df_shifted = pd.concat(frames_shifted, ignore_index=True)
    df_shifted = df_shifted.dropna(subset=['power_target']).reset_index(drop=True)
    ts_shifted = df_shifted['timestamp']

    fold_metrics_all = []

    for t_train_end, t_val_end in folds:
        mask_tr = ts_shifted <= t_train_end

        X_tr = df_shifted.loc[mask_tr, feature_cols_all]
        y_tr = df_shifted.loc[mask_tr, 'power_target']

        if X_tr.empty:
            continue

        ct = ColumnTransformer([
            ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
             [cat_idx])
        ], remainder='passthrough')
        pipe = Pipeline([
            ('prep', ct),
            ('scaler', StandardScaler()),
            ('model', LinearRegression())
        ])
        pipe.fit(X_tr, y_tr)

        for house in HOUSES:
            mask_h = (ts_shifted > t_train_end) & \
                     (ts_shifted <= t_val_end) & \
                     (df_shifted['house_id'] == house)
            X_vl_h = df_shifted.loc[mask_h, feature_cols_all].head(horizon_points)
            y_vl_h = df_shifted.loc[mask_h, 'power_target'].head(horizon_points)
            ts_vl_h = df_shifted.loc[mask_h, 'timestamp'].head(horizon_points)

            if len(y_vl_h) < horizon_points:
                continue

            y_pred_h = pipe.predict(X_vl_h)
            fold_metrics_all.append({
                'house': house,
                **evaluate(y_vl_h.values, y_pred_h)
            })

            # сохраняем предсказания последнего фолда
            if t_train_end == folds[-1][0]:
                save_predictions(
                    ts=ts_vl_h.values,
                    house_ids=np.full(len(y_vl_h), house),
                    y_true=y_vl_h.values,
                    y_pred=y_pred_h,
                    horizon_name=horizon_name,
                    model_name='LinearRegression',
                    filename='predictions_lr.csv'
                )

    df_fold = pd.DataFrame(fold_metrics_all)
    for house in HOUSES:
        m = df_fold[df_fold['house'] == house]
        if len(m) == 0:
            continue
        lr_rows.append({
            'Модель': 'Linear Regression', 'Дом': house, 'Горизонт': horizon_name,
            'MAE': round(m['MAE'].mean(), 3),
            'RMSE': round(m['RMSE'].mean(), 3),
            'MAPE': round(m['MAPE'].mean(), 3),
        })

df_lr = pd.DataFrame(lr_rows)
df_lr.to_csv('results_lr.csv', index=False)

print('Linear Regression: MAPE:')
pivot = df_lr.pivot(index='Горизонт', columns='Дом', values='MAPE')
pivot = pivot.reindex(list(HORIZONS.keys()))
pivot['Среднее'] = pivot.mean(axis=1).round(2)
print(pivot.round(2).to_string())

LR: 100%|██████████| 6/6 [01:08<00:00, 11.44s/it]

Linear Regression: MAPE:
Дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
Горизонт                                                                                 
4h          22.36    29.32    34.18    25.97    23.28    20.74    23.26    22.00    25.14
8h          20.76    20.43    22.53    20.55    19.80    22.14    21.73    20.64    21.07
24h          6.88     8.21     9.81     6.77     8.08     5.90     6.30     6.80     7.34
7d           5.92     7.76    10.21     5.79     7.27     6.48     4.37     5.96     6.72
14d          6.04     9.86    12.57     6.20     7.74     7.34     5.15     6.55     7.68
1m          11.46    19.57    27.52    11.84    14.90    12.09    10.16    12.10    14.96


In [12]:
# график LR: все горизонты, house_1, последний фолд
t_train_end, t_val_end = folds[-1]
house_plot = 'house_1'

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f'Горизонт {h}' for h in horizons.keys()],
    vertical_spacing=0.1,
    horizontal_spacing=0.08
)

for i, (horizon_name, horizon_points) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    # сдвигаем цель
    df_h = df_long[df_long['house_id'] == house_plot].copy()
    df_h['power_target'] = df_h['power'].shift(-horizon_points)
    df_h = df_h.dropna(subset=['power_target'])
    ts_h = df_h['timestamp']

    mask_tr = ts_h <= t_train_end
    mask_vl = (ts_h > t_train_end) & (ts_h <= t_val_end)

    X_tr = df_h.loc[mask_tr, feature_cols_all]
    y_tr = df_h.loc[mask_tr, 'power_target']

    X_vl = df_h.loc[mask_vl, feature_cols_all].head(horizon_points)
    y_vl = df_h.loc[mask_vl, 'power_target'].head(horizon_points)
    ts_vl = df_h.loc[mask_vl, 'timestamp'].head(horizon_points)

    if len(y_vl) < horizon_points:
        continue

    ct = ColumnTransformer([
        ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
         [cat_idx])
    ], remainder='passthrough')
    pipe = Pipeline([
        ('prep', ct),
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ])
    pipe.fit(X_tr, y_tr)
    y_pred = pipe.predict(X_vl)

    fig.add_trace(go.Scatter(
        x=ts_vl, y=y_vl.values,
        mode='lines', name='факт',
        line=dict(color='#1f77b4', width=1.2, shape='hv'),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=ts_vl, y=y_pred,
        mode='lines', name='Прогноз LR',
        line=dict(color='#d62728', width=1.2, shape='hv'),
        showlegend=(i == 0)
    ), row=row, col=col)

fig.update_layout(
    title='Linear Regression: Прогнозные и фактические значения '
          'по всем горизонтам прогнозирования (дом 1, fold 5)',
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9))
)

fig.show()
fig.write_image('29_lr_forecast_all_horizons.png', scale=2)

In [13]:
param_grids = {
    'Ridge': {'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]},
    'Lasso': {'model__alpha': [0.001, 0.01, 0.1, 1.0, 10.0]},
    'ElasticNet': {'model__alpha': [0.001, 0.01, 0.1, 1.0],
                   'model__l1_ratio': [0.2, 0.5, 0.8]},
}

model_classes = {
    'Ridge': Ridge(max_iter=5000),
    'Lasso': Lasso(max_iter=5000),
    'ElasticNet': ElasticNet(max_iter=5000),
}

reg_rows = []

for model_name, base_model in tqdm(model_classes.items(), desc='Регуляризованные LR'):

    for horizon_name, horizon_points in tqdm(
        HORIZONS.items(), desc=f'{model_name}', leave=False
    ):
        frames_shifted = []
        for house in HOUSES:
            df_h = df_long[df_long['house_id'] == house].copy()
            df_h['power_target'] = df_h['power'].shift(-horizon_points)
            frames_shifted.append(df_h)

        df_shifted = pd.concat(frames_shifted, ignore_index=True)
        df_shifted = df_shifted.dropna(subset=['power_target']).reset_index(drop=True)
        ts_shifted = df_shifted['timestamp']

        # подбор alpha на среднем фолде
        t_train_end_gs, t_val_end_gs = folds[2]
        mask_tr_gs = ts_shifted <= t_train_end_gs
        X_tr_gs = df_shifted.loc[mask_tr_gs, feature_cols_all]
        y_tr_gs = df_shifted.loc[mask_tr_gs, 'power_target']

        ct_gs = ColumnTransformer([
            ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
             [cat_idx])
        ], remainder='passthrough')
        pipe_gs = Pipeline([
            ('prep', ct_gs),
            ('scaler', StandardScaler()),
            ('model', base_model)
        ])
        gs = GridSearchCV(
            pipe_gs, param_grids[model_name],
            cv=3, scoring='neg_mean_absolute_error',
            n_jobs=-1, refit=True
        )
        gs.fit(X_tr_gs, y_tr_gs)
        best_params = gs.best_params_
        best_params_clean = {k.replace('model__', ''): v
                             for k, v in best_params.items()}
        print(f'{model_name} | {horizon_name}: {best_params_clean}')

        fold_metrics_all = []

        for t_train_end, t_val_end in folds:
            mask_tr = ts_shifted <= t_train_end
            X_tr = df_shifted.loc[mask_tr, feature_cols_all]
            y_tr = df_shifted.loc[mask_tr, 'power_target']

            if X_tr.empty:
                continue

            ct = ColumnTransformer([
                ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
                 [cat_idx])
            ], remainder='passthrough')
            pipe = Pipeline([
                ('prep', ct),
                ('scaler', StandardScaler()),
                ('model', base_model.set_params(**best_params_clean))
            ])
            pipe.fit(X_tr, y_tr)

            for house in HOUSES:
                mask_h = (ts_shifted > t_train_end) & \
                         (ts_shifted <= t_val_end) & \
                         (df_shifted['house_id'] == house)
                X_vl_h = df_shifted.loc[mask_h, feature_cols_all].head(horizon_points)
                y_vl_h = df_shifted.loc[mask_h, 'power_target'].head(horizon_points)
                ts_vl_h = df_shifted.loc[mask_h, 'timestamp'].head(horizon_points)

                if len(y_vl_h) < horizon_points:
                    continue

                y_pred_h = pipe.predict(X_vl_h)
                fold_metrics_all.append({
                    'house': house,
                    **evaluate(y_vl_h.values, y_pred_h)
                })

                if t_train_end == folds[-1][0]:
                    save_predictions(
                        ts=ts_vl_h.values,
                        house_ids=np.full(len(y_vl_h), house),
                        y_true=y_vl_h.values,
                        y_pred=y_pred_h,
                        horizon_name=horizon_name,
                        model_name=model_name,
                        filename=f'predictions_{model_name.lower()}.csv'
                    )

        df_fold = pd.DataFrame(fold_metrics_all)
        for house in HOUSES:
            m = df_fold[df_fold['house'] == house]
            if len(m) == 0:
                continue
            reg_rows.append({
                'Модель': model_name,
                'Дом': house,
                'Горизонт': horizon_name,
                'MAE': round(m['MAE'].mean(), 3),
                'RMSE': round(m['RMSE'].mean(), 3),
                'MAPE': round(m['MAPE'].mean(), 3),
                'best_params': str(best_params_clean),
            })

df_reg = pd.DataFrame(reg_rows)
df_reg.to_csv('results_regularized_lr.csv', index=False)

for model_name in ['Ridge', 'Lasso', 'ElasticNet']:
    subset = df_reg[df_reg['Модель'] == model_name]
    pivot = subset.pivot(index='Горизонт', columns='Дом', values='MAPE')
    pivot = pivot.reindex(list(HORIZONS.keys()))
    pivot['Среднее'] = pivot.mean(axis=1).round(2)
    print(f'\n{model_name}: MAPE:')
    print(pivot.round(2).to_string())

Ridge:   0%|          | 0/6 [00:00<?, ?it/s]

Ridge | 4h: {'alpha': 10.0}



Ridge:  17%|█▋        | 1/6 [00:27<02:16, 27.25s/it]

Ridge | 8h: {'alpha': 10.0}



Ridge:  33%|███▎      | 2/6 [00:52<01:44, 26.24s/it]

...

ElasticNet | 14d: {'alpha': 0.1, 'l1_ratio': 0.8}



ElasticNet:  83%|████████▎ | 5/6 [1:00:15<10:19, 619.93s/it]

ElasticNet | 1m: {'alpha': 0.1, 'l1_ratio': 0.8}



Регуляризованные LR: 100%|██████████| 3/3 [1:38:32<00:00, 1970.88s/it]


Ridge: MAPE:
Дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
Горизонт                                                                                 
4h          21.25    27.10    30.79    24.46    21.24    19.72    22.77    20.44    23.47
8h          19.39    17.92    18.85    18.70    17.21    21.13    20.94    18.80    19.12
24h          7.00     8.34     9.99     6.88     7.99     5.90     6.36     6.68     7.39
7d           5.86     7.18     9.00     5.72     7.27     6.55     4.49     5.61     6.46
14d          5.91     9.96    12.67     6.36     8.25     7.15     5.20     6.84     7.79
1m          11.49    19.36    27.42    12.21    14.78    12.02    10.24    12.09    14.95

Lasso: MAPE:
Дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
Горизонт                                                                                 
4h          20.82    26.56    30.33    24.24    20.78    19.28    22.73 

In [14]:
if 'df_naive' not in dir():
    df_naive = pd.read_csv('results_naive.csv')
if 'df_seasonal' not in dir():
    df_seasonal = pd.read_csv('results_seasonal_naive.csv')
    df_seasonal.columns = ['Модель', 'Дом', 'Горизонт', 'MAE', 'RMSE', 'MAPE']
if 'df_lr' not in dir():
    df_lr = pd.read_csv('results_lr.csv')
    df_lr.columns = ['Модель', 'Дом', 'Горизонт', 'MAE', 'RMSE', 'MAPE']

df_all = pd.concat([df_naive, df_seasonal, df_lr, df_reg], ignore_index=True)
df_all.to_csv('results_classical.csv', index=False)

print('Среднее MAPE по всем домам:')
summary = df_all.groupby(['Модель', 'Горизонт'])['MAPE'].mean().unstack()
summary = summary[list(HORIZONS.keys())]
print(summary.round(2).to_string())

Среднее MAPE по всем домам:
Горизонт              4h     8h    24h     7d    14d     1m
Модель                                                     
ElasticNet         23.47  19.09   7.42   6.52   7.91  14.56
Lasso              23.10  19.27   7.36   6.48   7.91  13.08
Linear Regression  25.14  21.07   7.34   6.72   7.68  14.96
Naive              17.43  33.68  34.93  37.18  37.01  37.72
Ridge              23.47  19.12   7.39   6.46   7.79  14.95
Seasonal Naive      7.43   7.22   7.30   7.65   7.75   8.40
